**Objetivo de aprendizaje:**  Identificar los filtros que componen un arreglo de filtros Bayer y resolver la tarea de interpolación o demosaicking con algoritmos de interpolación del estado del arte.

Implementar un sistema de fotografía computacional en python, fotografía en color usando el filtro Bayer visto en clase para generar el mosaico (medida) https://en.wikipedia.org/wiki/Bayer_filter, seleccionar algoritmos de interpolación 2D, por ejemplo, interpolación bicúbica que le permitan interpolar cada uno de los canales.

Para la tarea 1 usar todo dataset Kodak que se compone de 24 imágenes RGB 

https://github.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite.

Usar métricas de calidad como el PSNR, SSIM y SAM para medir la calidad. 

https://www.mathworks.com/help/images/ref/psnr.html

https://www.mathworks.com/help/images/ref/ssim.html

https://www.mathworks.com/help/images/ref/sam.html 


Tabular las métricas de calidad para cada imagen y el promedio y la desviación estándar para todo el dataset.

Bonus: Implementar este algoritmo de demosaicking 

https://web.stanford.edu/class/ee367/reading/Demosaicing_ICASSP04.pdf

In [2]:
#Librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from sewar.full_ref import sam

#Funciones auxiliares

def ShowImg(Img, Recons):
    fig, ax = plt.subplots(1, 2, figsize=(18, 9))

    ax[0].imshow(Img)
    ax[0].set_title("Original")
    ax[0].set_xticks([])
    ax[0].set_yticks([])

    ax[1].imshow(Recons)
    ax[1].set_title("Reconstruida")
    ax[1].set_xticks([])
    ax[1].set_yticks([])

    plt.show()

def ShowMatriz(matriz):
    rows = matriz.shape[0]
    columns = matriz.shape[1]
    for i in range(rows):
        for j in range(columns):
            print(f"{matriz[i][j]:.2f}", end=" ")
        print()

In [3]:
def bilinearInterpolationG(Tmatriz,i,j):
    i = i + 2
    j = j + 2
    Value = 1/4 * (Tmatriz[i-1][j] + Tmatriz[i+1][j] + Tmatriz[i][j-1] +  Tmatriz[i][j+1])
    return Value

def bilinearInterpolationRBH(Tmatriz,i,j):
    i = i + 2
    j = j + 2
    Value = 1/2 * (Tmatriz[i][j-1] + Tmatriz[i][j+1])
    return Value

def bilinearInterpolationRBV(Tmatriz,i,j):
    i = i + 2
    j = j + 2
    Value = 1/2 * (Tmatriz[i-1][j] + Tmatriz[i+1][j])
    return Value

def bilinearInterpolationRB(Tmatriz,i,j):
    i = i + 2
    j = j + 2
    Value = 1/4 * (Tmatriz[i-1][j-1] + Tmatriz[i+1][j-1] + Tmatriz[i-1][j+1] +  Tmatriz[i+1][j+1])
    return Value

def Gradient(Tmatriz,i,j):
    i = i + 2
    j = j + 2
    Value = Tmatriz[i][j] - 1/4 * (Tmatriz[i-2][j] + Tmatriz[i+2][j] + Tmatriz[i][j-2] + Tmatriz[i][j+2])
    return Value

In [4]:
def BilinearInterpolation(Y,RedMask,GreenMask,BlueMask):
    TRed   = np.array((RedMask   * Y), dtype=float)
    TGreen = np.array((GreenMask * Y), dtype=float)
    TBlue  = np.array((BlueMask  * Y), dtype=float)

    rows = TGreen.shape[0]
    columns = TGreen.shape[1]

    for i in range(rows):
        for j in range(columns):
            if GreenMask[i][j] == 0: # Recorremos la matriz verde
                #Manejar esquinas
                if i == 0 and j == 0:
                    TGreen[i][j] = 1/2 * (TGreen[i+1][j] + TGreen[i][j+1])
                elif i == 0 and j == columns -1:
                    TGreen[i][j] = 1/2 * (TGreen[i+1][j] + TGreen[i][j-1])
                elif i == (rows - 1) and j == 0:
                    TGreen[i][j] = 1/2 * (TGreen[i][j+1] + TGreen[i-1][j])
                elif i == (rows - 1) and j == (columns - 1):
                    TGreen[i][j] = 1/2 * (TGreen[i][j-1] + TGreen[i-1][j])
                #Manejar bordes    
                elif i == 0:
                    TGreen[i][j] = 1/3 * (TGreen[i][j-1]  + TGreen[i+1][j]  + TGreen[i][j+1])
                elif i == rows - 1:
                    TGreen[i][j] = 1/3 * (TGreen[i][j-1]  + TGreen[i-1][j]  + TGreen[i][j+1])
                elif j == 0:
                    TGreen[i][j] = 1/3 * (TGreen[i-1][j]  + TGreen[i][j+1]  + TGreen[i+1][j])
                elif j == columns - 1:
                    TGreen[i][j] = 1/3 * (TGreen[i-1][j]  + TGreen[i][j-1]  + TGreen[i+1][j])
                #Manejar general 
                else:
                    TGreen[i][j] = 1/4 * (TGreen[i+1][j]  + TGreen[i-1][j]  + TGreen[i][j+1] + TGreen[i][j-1])

            if RedMask[i][j] == 0: # Recorremos la matriz roja
                if BlueMask[i][j] == 0: #Posicion verde
                    #Casos de borde
                    if j == (columns - 1) and j%2 == 1:
                        TRed[i][j] = TRed[i][j-1]
                    elif i == (rows - 1)  and i%2 == 1:
                        TRed[i][j] = TRed[i-1][j]
                    #Manejar General
                    else:
                        if i%2 == 0:
                            TRed[i][j] = 1/2 * (TRed[i][j-1] + TRed[i][j+1])
                        else:
                            TRed[i][j] = 1/2 * (TRed[i-1][j] + TRed[i+1][j])
                        
                else:                 #Posicion azul
                    #Casos de borde
                    if i == (rows - 1) and j == (columns -1):
                        TRed[i][j] = TRed[i-1][j-1] 
                    elif j == (columns - 1):
                        TRed[i][j] = 1/2 * (TRed[i-1][j-1] + TRed[i+1][j-1])
                    elif i == (rows - 1):
                        TRed[i][j] = 1/2 * (TRed[i-1][j-1] + TRed[i-1][j+1])
                    #Manejar general 
                    else:
                        TRed[i][j] = 1/4 * (TRed[i-1][j-1] + TRed[i+1][j+1] + TRed[i+1][j-1] + TRed[i-1][j+1])

            if BlueMask[i][j] == 0: # Recorremos la matriz azul
                if RedMask[i][j] == 0: #Posicion verde
                    if j == 0:
                        TBlue[i][j] = TBlue[i][j+1]
                    elif i == 0:
                        TBlue[i][j] = TBlue[i+1][j]
                    #Manejar general 
                    else:
                        if i%2 == 0:
                            TBlue[i][j] = 1/2 * (TBlue[i-1][j] + TBlue[i+1][j])
                        else:
                            TBlue[i][j] = 1/2 * (TBlue[i][j-1] + TBlue[i][j+1])

                else:                 #Posicion Roja
                    #Casos de borde
                    if i == 0 and j == 0:
                        TBlue[i][j] = TBlue[i+1][j+1]
                    elif j == 0:
                        TBlue[i][j] = 1/2 * (TBlue[i-1][j+1] + TBlue[i+1][j+1])
                    elif i == 0:
                        TBlue[i][j] = 1/2 * (TBlue[i+1][j-1] + TBlue[i+1][j+1])
                    #Manejar general 
                    else:
                        TBlue[i][j] = 1/4 * (TBlue[i-1][j-1] + TBlue[i+1][j+1] + TBlue[i+1][j-1] + TBlue[i-1][j+1])

    Img= np.zeros((rows, columns, 3), dtype=float) #Imagen Reconstruida
    Img[:,:,0] = TRed
    Img[:,:,1] = TGreen
    Img[:,:,2] = TBlue 
    return Img

In [6]:
def MeasurementModel(imgInput):
    rows   = imgInput.shape[0]  #Filas
    columns = imgInput.shape[1] #Columnas

    B0 = np.array([[1,0],   # Rojo
                   [0,0]])
    
    B1 = np.array([[0,1],   # Verde
                   [1,0]]) 

    B2 = np.array([[0,0],   # Azul
                   [0,1]])

    MatrixOnes = np.ones((rows//2,columns//2),dtype=float)

    RedMask    = np.kron(MatrixOnes,B0)
    GreenMask  = np.kron(MatrixOnes,B1)
    BlueMask   = np.kron(MatrixOnes,B2)

    YBlue  = BlueMask  * imgInput[:,:,2]
    YGreen = GreenMask * imgInput[:,:,1]
    YRed   = RedMask   * imgInput[:,:,0]

    Y = YBlue + YGreen + YRed
    return Y,RedMask,GreenMask,BlueMask

def ImprovedBilinearInterpolation(Y,RedMask,GreenMask,BlueMask):
    # Lectura
    TRed   = np.array((RedMask   * Y), dtype=float)
    TGreen = np.array((GreenMask * Y), dtype=float)
    TBlue  = np.array((BlueMask  * Y), dtype=float)

    # Escritura
    RTRed   = np.array(TRed) 
    RTGreen = np.array(TGreen)
    RTBlue  = np.array(TBlue)

    TRed = np.pad(TRed, pad_width=2, mode='constant', constant_values=0)
    TGreen = np.pad(TGreen, pad_width=2, mode='constant', constant_values=0)
    TBlue = np.pad(TBlue, pad_width=2, mode='constant', constant_values=0)

    alfa= 1/2
    beta = 5/8 
    gamma = 3/4

    rows = RTGreen.shape[0]
    columns = RTGreen.shape[1]

    for i in range(rows):
        for j in range(columns):
            if GreenMask[i][j] == 0: # Recorremos la matriz verde
                if RedMask[i][j] == 1:
                    RTGreen[i][j] = bilinearInterpolationG(TGreen,i,j) + alfa*Gradient(TRed,i,j)
                elif BlueMask[i][j] == 1:
                    RTGreen[i][j] = bilinearInterpolationG(TGreen,i,j) + alfa*Gradient(TBlue,i,j)
                

            if RedMask[i][j] == 0: # Recorremos la matriz roja
                if BlueMask[i][j] == 1:
                    RTRed[i][j] =  bilinearInterpolationRB(TRed,i,j) + gamma*Gradient(TBlue,i,j)
                elif GreenMask[i][j] == 1:
                    if i%2 == 0:
                        RTRed[i][j] =  bilinearInterpolationRBH(TRed,i,j) + beta*Gradient(TGreen,i,j)
                    else:
                        RTRed[i][j] =  bilinearInterpolationRBV(TRed,i,j) + beta*Gradient(TGreen,i,j)

            if BlueMask[i][j] == 0:  # Recorremos la matriz azul
                if RedMask[i][j] == 1:
                    RTBlue[i][j] =  bilinearInterpolationRB(TBlue,i,j) + gamma*Gradient(TRed,i,j)
                elif GreenMask[i][j] == 1:
                    if i%2 == 1:
                        RTBlue[i][j] =  bilinearInterpolationRBH(TBlue,i,j) + beta*Gradient(TGreen,i,j)
                    else:
                        RTBlue[i][j] =  bilinearInterpolationRBV(TBlue,i,j) + beta*Gradient(TGreen,i,j)

                
    Img= np.zeros((rows, columns, 3), dtype=float) #Imagen Reconstruida
    Img[:,:,0] = RTRed
    Img[:,:,1] = RTGreen
    Img[:,:,2] = RTBlue 
    return Img

def GetMetrics(Original,Recons):
    Recons_PSNR = psnr(Original,Recons)
    Recons_SSIM = ssim(Original,Recons,channel_axis=2,data_range=2**24-1)
    Recons_SAM  = sam(Original,Recons)

    return Recons_PSNR,Recons_SSIM, Recons_SAM
    

In [7]:
def ProcessImages(interpolationMethod):
    metricsTable = pd.DataFrame(columns=["Imagen","PSNR","SSIM","SAM"])
    stadicsTable = pd.DataFrame({"Estadistica":["PSNR","SSIM","SAM"]})
    for i in range(1,25):
        if i < 10:
            index = "0" + str(i)
        else:
            index = str(i)
        
        imgName = f"Kodak-Lossless-True-Color-Image-Suite/PhotoCD_PCD0992/{index}.png"
        imgInput = plt.imread(imgName).astype(float)
        Y,RedMask,GreenMask,BlueMask = MeasurementModel(imgInput)
        Reconstructed = interpolationMethod(Y,RedMask,GreenMask,BlueMask)
    # ShowMatriz(Reconstructed[:,:,2])
    # ShowImg(imgInput,Reconstructed)
        Recons_PSNR,Recons_SSIM, Recons_SAM = GetMetrics(imgInput,Reconstructed)
        metricsTable.loc[i-1] = [index,Recons_PSNR,Recons_SSIM,Recons_SAM]

    PSNR_mean = metricsTable["PSNR"].mean()
    SSIM_mean = metricsTable["SSIM"].mean()
    SAM_mean  = metricsTable["SAM"].mean()

    stadicsTable["Promedio"] = [PSNR_mean,SSIM_mean,SAM_mean]

    PSNR_std  = metricsTable["PSNR"].std()
    SSIM_std  = metricsTable["SSIM"].std()
    SAM_std  = metricsTable["SAM"].std()

    stadicsTable["Desviacion Estandar"] = [PSNR_std,SSIM_std,SAM_std]

    return metricsTable, stadicsTable



In [8]:
TableBilinear, StadicsBilinear = ProcessImages(BilinearInterpolation)

print(TableBilinear)
print(StadicsBilinear)

   Imagen       PSNR  SSIM       SAM
0      01  26.024862   1.0  0.111993
1      02  32.066357   1.0  0.101178
2      03  32.906393   1.0  0.053818
3      04  32.570901   1.0  0.054861
4      05  26.467717   1.0  0.128899
5      06  27.402535   1.0  0.077704
6      07  32.379626   1.0  0.054225
7      08  23.601422   1.0  0.120535
8      09  31.732841   1.0  0.047743
9      10  31.654734   1.0  0.052199
10     11  28.940616   1.0  0.090225
11     12  32.148317   1.0  0.038057
12     13  23.857779   1.0  0.143095
13     14  28.788873   1.0  0.092059
14     15  30.669503   1.0  0.053984
15     16  30.456001   1.0  0.068227
16     17  31.976231   1.0  0.070544
17     18  27.924935   1.0  0.137307
18     19  28.163663   1.0  0.080756
19     20  29.857372   1.0  0.041848
20     21  28.339539   1.0  0.078073
21     22  30.240109   1.0  0.067135
22     23  33.499575   1.0  0.045449
23     24  26.679257   1.0  0.099201
  Estadistica   Promedio  Desviacion Estandar
0        PSNR  29.514548     

In [9]:
TableImproved, StadicsImproved = ProcessImages(ImprovedBilinearInterpolation)

print(TableImproved)
print(StadicsImproved)

   Imagen       PSNR  SSIM       SAM
0      01  29.829328   1.0  0.070790
1      02  34.954386   1.0  0.076136
2      03  36.048901   1.0  0.037694
3      04  36.065704   1.0  0.036534
4      05  30.714040   1.0  0.076713
5      06  30.986184   1.0  0.050810
6      07  35.771423   1.0  0.036631
7      08  26.985637   1.0  0.079536
8      09  35.119764   1.0  0.031650
9      10  35.586365   1.0  0.032411
10     11  32.714620   1.0  0.057552
11     12  35.537230   1.0  0.025644
12     13  28.150088   1.0  0.085769
13     14  32.070369   1.0  0.063032
14     15  33.879583   1.0  0.037190
15     16  33.828254   1.0  0.045626
16     17  35.034249   1.0  0.048263
17     18  31.622573   1.0  0.087115
18     19  31.189827   1.0  0.055559
19     20  33.582260   1.0  0.026848
20     21  31.787431   1.0  0.051270
21     22  33.217198   1.0  0.047122
22     23  37.127800   1.0  0.030486
23     24  29.726176   1.0  0.068413
  Estadistica   Promedio  Desviacion Estandar
0        PSNR  32.980391     